# Séance 5 — ML non supervisé (clustering, PCA)

**Decision problem:** how do we detect useful patterns when labels do not exist?

Official topic preserved: clustering and PCA. The output is a segment table for interpretation.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
SNAPSHOT = "iot_FR_today_5-y_chatgpt_iphone_meteo.csv"
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "data" / "snapshots" / SNAPSHOT).exists():
        ROOT = candidate
        break
OUT = ROOT / "outputs"
for d in [OUT / "bloc1", OUT / "bloc2", OUT / "bloc3", OUT / "final_product"]:
    d.mkdir(parents=True, exist_ok=True)
DATA = ROOT / "data" / "snapshots"

def load_raw_trends():
    p = DATA / "iot_FR_today_5-y_chatgpt_iphone_meteo.csv"
    df = pd.read_csv(p)
    df["date"] = pd.to_datetime(df["date"])
    for c in ["chatgpt", "iphone", "meteo"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df.sort_values("date").reset_index(drop=True)

def load_clean_long():
    p = OUT / "bloc1" / "clean_trends_long.csv"
    if p.exists():
        df = pd.read_csv(p, parse_dates=["date"])
    else:
        raw = load_raw_trends()
        df = raw.melt(id_vars="date", var_name="signal", value_name="interest")
    return df.sort_values(["date", "signal"]).reset_index(drop=True)

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [ ]:
wide = pd.read_csv(OUT / "bloc1" / "clean_trends_wide.csv", parse_dates=["date"]) if (OUT / "bloc1" / "clean_trends_wide.csv").exists() else load_raw_trends()
X = wide[["chatgpt", "iphone", "meteo"]]
scaled = StandardScaler().fit_transform(X)
wide["attention_regime"] = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(scaled)
regimes = wide.groupby("attention_regime")[["chatgpt", "iphone", "meteo"]].mean().round(2)
wide[["date", "chatgpt", "iphone", "meteo", "attention_regime"]].to_csv(OUT / "bloc1" / "attention_regimes.csv", index=False)
regimes.to_csv(OUT / "bloc1" / "attention_regime_profiles.csv")
regimes

## Practical exercise

Rename each regime with a business interpretation.

## Conclusion

Unsupervised learning creates hypotheses for action; it does not prove segments are real.